## 准备数据

In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [2]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [3]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        self.W1 = tf.Variable(tf.random.normal([784, 256], stddev=0.1))
        self.b1 = tf.Variable(tf.zeros([256]))
        self.W2 = tf.Variable(tf.random.normal([256, 10], stddev=0.1))
        self.b2 = tf.Variable(tf.zeros([10]))
        ####################
    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
         # 将输入的28x28图像展平为784维向量
        x = tf.reshape(x, [-1, 784])
        h1 = tf.nn.relu(tf.matmul(x, self.W1) + self.b1)
        logits = tf.matmul(h1, self.W2) + self.b2
        ####################
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [6]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    optimizer.apply_gradients(zip(grads, trainable_vars))

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [7]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 1.6755689 ; accuracy 0.48668334
epoch 1 : loss 1.49087 ; accuracy 0.5732333
epoch 2 : loss 1.3308822 ; accuracy 0.6404333
epoch 3 : loss 1.1926526 ; accuracy 0.69115
epoch 4 : loss 1.0738747 ; accuracy 0.7310333
epoch 5 : loss 0.9722214 ; accuracy 0.7604
epoch 6 : loss 0.8855098 ; accuracy 0.7834333
epoch 7 : loss 0.8115532 ; accuracy 0.79855
epoch 8 : loss 0.74837554 ; accuracy 0.81131667
epoch 9 : loss 0.69429797 ; accuracy 0.82238334
epoch 10 : loss 0.6478373 ; accuracy 0.83165
epoch 11 : loss 0.6077744 ; accuracy 0.8398333
epoch 12 : loss 0.5731097 ; accuracy 0.8476667
epoch 13 : loss 0.54297405 ; accuracy 0.8544667
epoch 14 : loss 0.5166537 ; accuracy 0.86015
epoch 15 : loss 0.49354428 ; accuracy 0.86468333
epoch 16 : loss 0.47314942 ; accuracy 0.86986667
epoch 17 : loss 0.4550676 ; accuracy 0.87408334
epoch 18 : loss 0.43894506 ; accuracy 0.878
epoch 19 : loss 0.42450503 ; accuracy 0.88128334
epoch 20 : loss 0.41150826 ; accuracy 0.8846167
epoch 21 : loss 0.3997368